# SEN2SR Lite — Google Colab Setup

This notebook is organized into separate, sequential code cells for mounting Google Drive, installing dependencies, loading the pretrained model, preparing a Sentinel-2 input, and running super-resolution.

**Before running:** update `DRIVE_FOLDER_PATH` and `INPUT_IMAGE_FILE` in the data-preparation cell after your dataset is available. The notebook uses dummy data only as a smoke test; it does not produce meaningful satellite super-resolution results until real, correctly ordered and preprocessed bands are supplied.

> Note: Confirm the model's expected input band order, scaling, spatial dimensions, and output scale against the SEN2SR Lite documentation before using real data.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Install dependencies
Run this cell once per fresh Colab runtime. If imports fail after installation, restart the runtime and continue from Cell 3.

In [ ]:
%pip install -q sen2sr mlstac rasterio git+https://github.com/ESDS-Leipzig/cubo.git

## Cell 3 — Imports and device check

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import rasterio
import mlstac

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

## Cell 4 — Download and load the pretrained SEN2SR Lite model

In [ ]:
MODEL_DIR = "/content/model/SEN2SRLite"
os.makedirs(MODEL_DIR, exist_ok=True)

mlstac.download(
    file="https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/main/mlm.json",
    output_dir=MODEL_DIR,
)

model = mlstac.load(MODEL_DIR).compiled_model(device=device)
model = model.to(device)
model.eval()

print("SEN2SR Lite model loaded.")

## Cell 5 — Configure your dataset paths

Edit these paths to match the files in your Google Drive. The input GeoTIFF must contain the model's expected bands in the documented order. A typical Sentinel-2 product may store bands in separate files, so a single `.tif` is only suitable if it already contains the required stacked bands.

In [ ]:
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/Colab Notebooks/Sentinel_Data"
INPUT_IMAGE_FILE = os.path.join(DRIVE_FOLDER_PATH, "your_input_image.tif")

print("Data folder exists:", os.path.isdir(DRIVE_FOLDER_PATH))
print("Input file exists:", os.path.isfile(INPUT_IMAGE_FILE))

## Cell 6 — Load and prepare a local GeoTIFF

This loader reads all bands from a multiband GeoTIFF, converts them to `float32`, and applies the example scaling factor of 10,000. Verify the product's stored value scale and the model's preprocessing requirements before using this normalization. This cell checks that there are 10 bands but cannot verify their semantic order.

In [ ]:
def load_and_prep_local_data(filepath, scale_factor=10_000):
    if not os.path.isfile(filepath):
        raise FileNotFoundError(
            f"Input file not found: {filepath}\n"
            "Update INPUT_IMAGE_FILE after uploading your dataset to Drive."
        )

    with rasterio.open(filepath) as src:
        image = src.read()  # (bands, height, width)
        print(f"Raster shape: {image.shape}")
        print(f"CRS: {src.crs}")
        print(f"Resolution: {src.res}")
        print(f"Data type: {image.dtype}")

    if image.ndim != 3 or image.shape[0] != 10:
        raise ValueError(
            f"Expected a 10-band GeoTIFF shaped (10, height, width); got {image.shape}. "
            "Stack/reorder the required bands according to the model documentation."
        )

    image = image.astype(np.float32) / scale_factor
    if not np.isfinite(image).all():
        raise ValueError("Input contains NaN or infinite values; clean/mask them before inference.")

    return torch.from_numpy(image).float().to(device)

# Uncomment once your real input file is ready:
# X = load_and_prep_local_data(INPUT_IMAGE_FILE)

# Temporary smoke-test tensor only — not real satellite data:
X = torch.rand(10, 128, 128, dtype=torch.float32, device=device)
print("Using dummy tensor for pipeline smoke test:", tuple(X.shape))

## Cell 7 — Run super-resolution

The original example assumes a 4× spatial scale factor. Confirm that this is the actual output scale and input contract of the loaded model. For real data, replace the dummy tensor in Cell 6 with `X = load_and_prep_local_data(INPUT_IMAGE_FILE)`.

In [ ]:
with torch.no_grad():
    superX = model(X.unsqueeze(0)).squeeze(0)

print("Input shape: ", tuple(X.shape))
print("Output shape:", tuple(superX.shape))

## Cell 8 — Visualize an RGB comparison

This visualization uses the band indices from the supplied example (`[9, 7, 5]`). Those indices must correspond to the intended RGB bands in your actual stacked tensor. Display scaling here is illustrative; adjust it for your data and avoid interpreting dummy-data imagery as a result.

In [ ]:
# Select a central crop. The output crop assumes a 4x scale factor.
lr_start, lr_stop = 32, min(96, X.shape[-1], X.shape[-2])
if lr_stop <= lr_start:
    raise ValueError("Input is too small for the selected crop.")

scale = 4
hr_start, hr_stop = lr_start * scale, lr_stop * scale

rgb_indices = [9, 7, 5]  # Verify these indices against your band stack.

original_rgb = X[rgb_indices, lr_start:lr_stop, lr_start:lr_stop]
output_rgb = superX[rgb_indices, hr_start:hr_stop, hr_start:hr_stop]

def display_rgb(tensor):
    arr = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    arr = np.clip(arr * 1.5, 0, 1)
    return arr

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(display_rgb(original_rgb))
axes[0].set_title("Input image (crop)")
axes[0].axis("off")

axes[1].imshow(display_rgb(output_rgb))
axes[1].set_title("SEN2SR Lite output (crop)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Cell 9 — Save output as a NumPy array (optional)

This saves the model output tensor as `.npy`. For a georeferenced output GeoTIFF, preserve and update the source raster's CRS and transform to match the model's output pixel size; do not simply copy the original transform if the spatial resolution changed.

In [ ]:
OUTPUT_NPY = os.path.join(DRIVE_FOLDER_PATH, "sen2sr_lite_output.npy")

# Uncomment after confirming X and superX contain real-data inference:
# os.makedirs(DRIVE_FOLDER_PATH, exist_ok=True)
# np.save(OUTPUT_NPY, superX.detach().cpu().numpy())
# print("Saved:", OUTPUT_NPY)

## Troubleshooting

- **CUDA out of memory:** use a smaller input tile and follow the model's recommended tiling/overlap procedure.
- **Model download or load errors:** check the SEN2SR Lite package/model documentation and the current `mlstac` API.
- **Unexpected output shape:** inspect `X.shape` and `superX.shape`; do not assume a 4× scale factor without verification.
- **Poor or meaningless results:** verify band order, reflectance scaling, nodata/cloud handling, input dimensions, and model-specific preprocessing.
- **Dummy data:** the random tensor only checks whether the pipeline executes. It is not a valid evaluation of model quality.